Grouping and Aggregating Data

Developing Applications with Apache Spark


databricks free

https://dbc-0b69ef93-5247.cloud.databricks.com/editor/notebooks/1577720052631600?o=7474657872577658#command/5993161087222598

## Grouping Data

Grouping is generally a precursor to aggregation:

In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, to_date

# 1. Criar um DataFrame de exemplo do zero
data = [
    ("Engineering", "New York", "2021-05-10", 120000, 50000),
    ("Engineering", "New York", "2022-03-15", 135000, 75000),
    ("HR", "London", "2020-01-12", 80000, 20000),
    ("HR", "London", "2023-07-20", 85000, 30000),
    ("Engineering", "San Francisco", "2021-11-05", 150000, 90000),
    ("Sales", "New York", "2022-06-18", 95000, 110000)
]

columns = ["department", "location", "hire_date", "salary", "revenue"]

# Criar o DataFrame e converter a coluna de data para o tipo Date
df = spark.createDataFrame(data, columns).withColumn("hire_date", to_date(col("hire_date")))

# Visualizar os dados iniciais
display(df)

department,location,hire_date,salary,revenue
Engineering,New York,2021-05-10,120000,50000
Engineering,New York,2022-03-15,135000,75000
HR,London,2020-01-12,80000,20000
HR,London,2023-07-20,85000,30000
Engineering,San Francisco,2021-11-05,150000,90000
Sales,New York,2022-06-18,95000,110000


In [0]:
# ==========================================
# 2. Basic grouping
# ==========================================
print("--- Basic grouping ---")
display(df.groupBy("department").count())

# ==========================================
# 3. Multi-column grouping
# ==========================================
print("--- Multi-column grouping ---")
display(df.groupBy("department", "location").avg("salary"))

# ==========================================
# 4. Using column expressions
# ==========================================
print("--- Using column expressions ---")
display(df.groupBy(col("department"), year("hire_date")).sum("revenue"))

--- Basic grouping ---


department,count
Engineering,3
HR,2
Sales,1


--- Multi-column grouping ---


department,location,avg(salary)
Engineering,New York,127500.0
HR,London,82500.0
Engineering,San Francisco,150000.0
Sales,New York,95000.0


--- Using column expressions ---


department,year(hire_date),sum(revenue)
Engineering,2021,140000
Engineering,2022,75000
HR,2020,20000
HR,2023,30000
Sales,2022,110000


##  Combining Multiple Aggregations

Multiple aggregations can be performed on the same GroupedData object using the `agg()` function:

In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg

# 1. Criar um DataFrame de exemplo com as colunas necessárias
data = [
    ("Engineering", 30, 120000),
    ("Engineering", 35, 135000),
    ("HR", 28, 80000),
    ("HR", 40, 85000),
    ("Engineering", 45, 150000),
    ("Sales", 26, 95000)
]

columns = ["department", "age", "salary"]
df = spark.createDataFrame(data, columns)

display(df)

department,age,salary
Engineering,30,120000
Engineering,35,135000
HR,28,80000
HR,40,85000
Engineering,45,150000
Sales,26,95000


In [0]:
# ==========================================
# 2. Using aggregate functions directly
# ==========================================
print("--- Using aggregate functions directly ---")
display(
    df.groupBy("department")
    .agg(
        sum("salary").alias("total_salary"),
        avg("age").alias("avg_age")
    )
)

# ==========================================
# 3. Alternative syntax using a dictionary
# ==========================================
print("--- Alternative syntax using a dictionary ---")
display(
    df.groupBy("department")
    .agg({
        "salary": "sum",
        "age": "avg"
    })
)

--- Using aggregate functions directly ---


department,total_salary,avg_age
Engineering,405000,36.666666666666664
HR,165000,34.0
Sales,95000,26.0


--- Alternative syntax using a dictionary ---


department,sum(salary),avg(age)
Engineering,405000,36.666666666666664
HR,165000,34.0
Sales,95000,26.0


## Window Functions

**Aggregation without losing row-level detail:**
* **What are they?** Special functions that perform calculations across a set of rows related to the current row
* **When to use:**
  * Calculating running totals and averages
  * Ranking records within groups
  * Accessing previous and next row values
  * Comparing row values to group aggregates

### Window Example
A window specification defines how rows are partitioned and ordered; window functions are then applied over it:

### groupBy vs Window
`groupBy()` collapses each group to a single output row. A window function adds the aggregate to every row, preserving the original detail - choose based on whether you still need the individual rows.

In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, sum, lag

# 1. Criar um DataFrame de exemplo do zero
data = [
    ("Engineering", 120000),
    ("Engineering", 135000),
    ("Engineering", 150000),
    ("HR", 80000),
    ("HR", 85000),
    ("Sales", 95000)
]

columns = ["department", "salary"]
df = spark.createDataFrame(data, columns)

display(df)

department,salary
Engineering,120000
Engineering,135000
Engineering,150000
HR,80000
HR,85000
Sales,95000


In [0]:
# 2. Window specification: define how rows are partitioned and ordered
windowSpec = Window.partitionBy("department").orderBy("salary")

# 3. Using window functions to preserve row-level detail while adding calculations
windowed_df = (
    df.withColumn("rank", rank().over(windowSpec))
      .withColumn("running_total", sum("salary").over(windowSpec))
      .withColumn("prev_salary", lag("salary").over(windowSpec))
)

display(windowed_df)

department,salary,rank,running_total,prev_salary
Engineering,120000,1,120000,null
Engineering,135000,2,255000,120000
Engineering,150000,3,405000,135000
HR,80000,1,80000,null
HR,85000,2,165000,80000
Sales,95000,1,95000,null


A função lag() no PySpark (e em SQL) serve para aceder a dados de linhas anteriores dentro da mesma partição de uma janela (Window).

Em termos práticos, ela permite "olhar para trás" num DataFrame ordenado para comparar o valor da linha atual com o valor de uma linha anterior.